# 05c_GMM_nivel1


In [1]:
import pandas as pd
import numpy as np
import time, joblib
from pathlib import Path
from datetime import datetime
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
DATA_MODELS = ROOT / 'data' / 'data_models_gustos'
INFORMES_BASE = ROOT / 'informes' / 'fase4_gustos' / '04_clustering' / 'nivel1'
DATA_MODELS.mkdir(parents=True, exist_ok=True)

MASTER_PATH = DATA_QC / 'features_tier1.parquet'
TIER_LEVEL = '1'

def reduce_high_card(df, max_unique=10):
    out = df.copy()
    for cat in out.select_dtypes(include=['object','category']).columns.tolist():
        if out[cat].nunique(dropna=True) > max_unique:
            top = out[cat].value_counts().head(max_unique).index.tolist()
            out[cat] = out[cat].where(out[cat].isin(top), 'OTHER')
    return out

def preprocess(df, nan_threshold_zero=0.30):
    df = df.drop(columns=['user_id']).copy()
    numeric = df.select_dtypes(include=['number']).columns.tolist()
    categorical = df.select_dtypes(include=['object','category']).columns.tolist()
    nan_pcts = df[numeric].isna().mean()
    num_low = nan_pcts[nan_pcts < nan_threshold_zero].index.tolist()
    num_high = nan_pcts[nan_pcts >= nan_threshold_zero].index.tolist()
    df = reduce_high_card(df, 10)
    pp = ColumnTransformer([
        ('num_low', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())]), num_low),
        ('num_high', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value=0)), ('sc', RobustScaler())]), num_high),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical),
    ])
    X = pp.fit_transform(df)
    return X, pp, pp.get_feature_names_out()

print(f'TIER {TIER_LEVEL}: cargando {MASTER_PATH.name}')
master = pd.read_parquet(MASTER_PATH)
user_ids = master['user_id'].values
print(f'  shape: {master.shape}')
X, preproc, names = preprocess(master)
joblib.dump(preproc, DATA_MODELS / f'preprocessor_tier{TIER_LEVEL}.joblib')
print(f'  X post-preproc: {X.shape}')
N = len(X)

from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
REPORT = INFORMES_BASE / '05c_gmm'
REPORT.mkdir(parents=True, exist_ok=True)
K_RANGE = list(range(2, 11))
SAMPLE_TRAIN = 60_000
SAMPLE_SIL = 20_000
results = []


TIER 1: cargando features_tier1.parquet
  shape: (114412, 79)


/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_6311/4275237113.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = df.select_dtypes(include=['object','category']).columns.tolist()
/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_6311/4275237113.py:22: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this w

  X post-preproc: (114412, 96)


In [2]:
rng = np.random.RandomState(42)
if len(X) > SAMPLE_TRAIN:
    train_idx = rng.choice(len(X), SAMPLE_TRAIN, replace=False)
    X_train = X[train_idx]
else:
    X_train = X
sil_idx = rng.choice(len(X), min(SAMPLE_SIL, len(X)), replace=False)
X_sil = X[sil_idx]
bics=[]; sils=[]
for k in K_RANGE:
    t0 = time.time()
    gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42, max_iter=200, n_init=1)
    gmm.fit(X_train)
    labels_all = gmm.predict(X)
    labels_sub = gmm.predict(X_sil)
    sil = float(silhouette_score(X_sil, labels_sub)) if len(set(labels_sub))>1 else float('nan')
    db = float(davies_bouldin_score(X, labels_all))
    bic = float(gmm.bic(X_train))
    unique, counts = np.unique(labels_all, return_counts=True)
    max_pct = float(counts.max()/len(labels_all)); min_pct = float(counts.min()/len(labels_all))
    bics.append(bic); sils.append(sil)
    results.append({'algorithm':'GMM','tier':TIER_LEVEL,'k':k,'silhouette':sil,'davies_bouldin':db,'bic':bic,'n_clusters_actual':int(len(set(labels_all))),'max_cluster_pct':max_pct,'min_cluster_pct':min_pct,'elapsed_s':time.time()-t0})
    print(f'  K={k}: sil={sil:.4f} db={db:.4f} bic={bic:.0f} max%={max_pct:.1%} min%={min_pct:.1%}')
    joblib.dump({'model':gmm,'labels':labels_all,'user_ids':user_ids}, DATA_MODELS / f'nivel{TIER_LEVEL}_gmm_k{k}.joblib')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_RANGE, bics, marker='o'); axes[0].set_xlabel('K'); axes[0].set_ylabel('BIC'); axes[0].grid(alpha=0.3); axes[0].set_title(f'BIC nivel{TIER_LEVEL}')
axes[1].plot(K_RANGE, sils, marker='o', color='C2'); axes[1].axhline(0.15, color='red', ls='--', alpha=0.4); axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette'); axes[1].grid(alpha=0.3); axes[1].set_title(f'Silhouette nivel{TIER_LEVEL}')
plt.tight_layout(); plt.savefig(REPORT / 'bic_silhouette.png', dpi=120, bbox_inches='tight'); plt.close()
pd.DataFrame(results).to_csv(REPORT / 'metrics.csv', index=False)
print('OK metrics.csv')


  K=2: sil=nan db=0.0027 bic=-70111 max%=100.0% min%=0.0%


  K=3: sil=0.6114 db=0.8061 bic=-9409325 max%=78.3% min%=0.0%


  K=4: sil=0.6104 db=0.7236 bic=-9381261 max%=78.3% min%=0.0%


  K=5: sil=0.6086 db=0.6090 bic=-9400247 max%=78.1% min%=0.0%


  K=6: sil=0.4375 db=0.6566 bic=-15816165 max%=65.6% min%=0.0%


  K=7: sil=0.4197 db=4.7809 bic=-18314471 max%=65.5% min%=0.0%


  K=8: sil=0.4330 db=2.7359 bic=-19482831 max%=64.6% min%=0.0%


  K=9: sil=0.1934 db=2.1019 bic=-22655515 max%=49.7% min%=0.0%


  K=10: sil=0.3341 db=4.0153 bic=-24201246 max%=59.2% min%=0.0%
OK metrics.csv
